In [24]:
import requests
import pandas as pd
import json
from upsetplot import UpSet, from_contents
import itertools
from functools import reduce
from pathlib import Path
import mygene
from io import StringIO
import time

In [2]:
# the query protein
query = "CADM4_HUMAN"

In [3]:
# Function that adds '_HUMAN' afterfix values

def add_suffix_human(value):
    return value + '_HUMAN'

In [4]:
# Function that takes a ppi dataframe and outputs from two columns only one column with the interactor of a specified single protein

def get_interactors_for_target(df, col_a, col_b, target_protein):    
    def get_interactors(row):
        if row[col_a] == target_protein:
            return row[col_b]
        elif row[col_b] == target_protein:
            return row[col_a]
        else:
            return None
    df['interactor_of_' + target_protein] = df.apply(get_interactors, axis = 1)
    
    return df

In [ ]:
# Function that uses the UniProt-API to convert the uniprotID to Protein name or vice versa

def convert_uniprotID_uniprotName(uniprotID_or_name): # e.g. can be P19320 or VCAM1_HUMAN

    uniprot_api_url = "https://rest.uniprot.org/uniprotkb" 
    format = "json"
    uniprot_request_url = f"{uniprot_api_url}/{uniprotID_or_name}?format={format}"
    
    try:
        uniprot_response = requests.get(uniprot_request_url)
        uniprot_response.raise_for_status()
        
        uniprot_json = uniprot_response.text
        uniprot_dict = json.loads(uniprot_json) # convering to dictionary

        if "_" in uniprotID_or_name:
            value = uniprot_dict["primaryAccession"]
        else:
            value = uniprot_dict['uniProtkbId']
        
        return (value)
    
    except requests.exceptions.RequestException as e:
        print(f"Request error: {e}")
        
    except json.JSONDecodeError as e:
        print(f"JSON decoding error: {e}")
        
    return None

# example
id_or_name = convert_uniprotID_uniprotName("NaN")
print(id_or_name)

In [54]:
# ID mapping from the Uniprot-API is used to batch retrieve uniprotIDs or uniprotNames

# this function extracts only the main protein & not the isoforms
def extract_main_protein(isoform_list):
    main_proteins = set()
    pattern = r'([A-Z0-9]+)-?\d*'
    
    for protein in isoform_list:
        match = re.match(pattern, protein)
        if match:
            protein_id = match.group(1)
            main_proteins.add(protein_id)
    main_proteins_list = list(main_proteins)
    
    return main_proteins_list


dummy_data = {'Name':['Karan','Rohit','Sahil','Aryan'],'Protein':["P58215", "Q96JB6", "Q93ZB1", "P14946"]}
dummy_df = pd.DataFrame(dummy_data)


list_of_uniprotIDs = dummy_df["Protein"].tolist()
ls = ",".join(list_of_uniprotIDs)
print(ls)

r = requests.post("https://rest.uniprot.org/idmapping/run", data={
    "from": "UniProtKB_AC-ID",
    "to": "UniProtKB-Swiss-Prot", 
    "ids": ls 
})

job_id = r.json()["jobId"]
print(job_id)


while True:
    response = requests.get(f"https://rest.uniprot.org/idmapping/status/{job_id}")
    data_json = json.loads(response.text)
   
    if "jobStatus" in data_json:
        job_status = data_json["jobStatus"]
        print(f"Job Status: {job_status}")
        
        if job_status != "RUNNING":
            break
 
    if "results" in data_json:
        if "_" in ls:
            desired_id = []
            for entry in data_json["results"]:
                if entry["from"] in list_of_uniprotIDs:
                    desired_id.append(entry["to"]["primaryAccession"])
            desired_id_proteins = extract_main_protein(desired_id)
            print(desired_id_proteins)

        else:
            desired_id = []
            for entry in data_json["results"]:
                if entry["from"] in list_of_uniprotIDs:
                    desired_id.append(entry["to"]["uniProtkbId"])
            print(desired_id)
        break

    time.sleep(5)

# TODO: make this function for a dataframe column (so input and output as dataframe column)

P58215,Q96JB6,Q93ZB1,P14946
7d96f42e23e41ae66e312fecc53850321b7cff8e
Job Status: RUNNING
['LOXL3_HUMAN', 'LOXL4_HUMAN', 'LOL1_ARATH', 'MPAL1_LOLPR']


In [52]:
import re

def extract_unique_base_ids(input_list):
    # Use a set to store unique base IDs
    unique_base_ids = set()

    # Regular expression pattern to match the base ID
    pattern = r'([A-Z0-9]+)-?\d*'

    # Iterate through the input list and extract the base IDs
    for item in input_list:
        match = re.match(pattern, item)
        if match:
            base_id = match.group(1)
            unique_base_ids.add(base_id)

    # Convert the set back to a list and return it
    unique_base_ids_list = list(unique_base_ids)
    return unique_base_ids_list

# Example usage:
input_list = ['Q9BY67-1', 'Q9BY67-3', 'Q9BY67-4', 'Q9BY67-5', 'Q9BY67-2', 'Q9BY67', 'Q13740-1', 'Q13740-2', 'Q13740-3', 'Q13740-4', 'Q13740', 'Q8NHJ6-1', 'Q8NHJ6-2', 'Q8NHJ6-3', 'Q8NHJ6']

unique_base_ids = extract_unique_base_ids(input_list)
print(unique_base_ids)


['Q13740', 'Q8NHJ6', 'Q9BY67']


In [ ]:
# this function will convert uniprotName to uniprotID

def convert_uniprotName_to_uniprotID(uniprotName):
    
    uniprot_api_url = "https://rest.uniprot.org/uniprotkb/search?query="
    fields = "accession,id"
    format = "tsv"
    request_url = f"{uniprot_api_url}{uniprotName}&fields={fields}&format={format}"
    
    try:
        uniprot_response = requests.get(request_url)
        uniprot_response.raise_for_status()
        
        tsv_data = uniprot_response.text
        df = pd.read_csv(StringIO(tsv_data), sep='\t')
        filtered_df = df[df["Entry Name"] == uniprotName]
        uniprotID = filtered_df.iloc[0,0]
    
        return (uniprotID)

    except requests.exceptions.RequestException as e:
        print(f"Request error: {e}")
    
    return None

# example
l = convert_uniprotName_to_uniprotID("ITB1_HUMAN")
print(l)



In [ ]:
# this function will convert uniprotID to uniprotName

def convert_uniprotID_to_uniprotName(uniprotID):

    if pd.isna(uniprotID):
        return None
    
    else:
        uniprot_api_url = "https://rest.uniprot.org/uniprotkb/search?query="
        fields = "accession,id"
        format = "tsv"
        request_url = f"{uniprot_api_url}{uniprotID}&fields={fields}&format={format}"
        
        try:
            uniprot_response = requests.get(request_url)
            uniprot_response.raise_for_status()
            
            tsv_data = uniprot_response.text
            df = pd.read_csv(StringIO(tsv_data), sep='\t')
            filtered_df = df[df["Entry"] == uniprotID]
            uniprotName = filtered_df.iloc[0,1]
        
            return (uniprotName)
        
        except requests.exceptions.RequestException as e:
            print(f"Request error: {e}")
        
        return None

# example
l = convert_uniprotID_to_uniprotName("P05556")
print(l)

In [ ]:
# Function that uses the MyGene-API to convert the entrezGeneID to UniprotID (this is for biogrid, because biogrid outputs only gene names)

def convert_geneID_uniprotID(geneID):

    mygene_api_url = "https://mygene.info/v3/gene"
    entrezGeneID = geneID
    mygene_request_url = f"{mygene_api_url}/{entrezGeneID}?fields=all&dotfield=false&size=10"

    mygene_response = requests.get(mygene_request_url)
    mygene_json = mygene_response.json()

    if "uniprot" in mygene_json:
        uniprot_id = mygene_json["uniprot"]["Swiss-Prot"]
    elif "pantherdb" in mygene_json:
        uniprot_id = mygene_json["pantherdb"]["uniprot_kb"]
    else:
        uniprot_id = "NA"
        
    return uniprot_id

# example
Uniprot_convert = convert_geneID_uniprotID(6047)
print(Uniprot_convert)

In [ ]:
# Function that uses the MyGene-API python package to convert multiple entrezGeneIDs to UniprotIDs from a dataframe column (this is for biogrid)
# This is an alternative to the function above, this uses the MyGene package instead querying the each input seperately through the API

def convert_geneIDs_uniprotIDs(df, df_column_name, df_new_column_name):

    mg = mygene.MyGeneInfo()
    
    # convert column to list
    list_of_column = df[df_column_name].tolist()
    
    # get the data from MyGene-API package
    df_mg = mg.getgenes(list_of_column, fields = 'uniprot', as_dataframe = True)
    
    # filter it, convert to list & add column to original df
    df_mg_filtered = df_mg[['uniprot.Swiss-Prot']]
    list_filtered = df_mg_filtered['uniprot.Swiss-Prot'].tolist()
    
    # append the list to the dataframe
    df[df_new_column_name] = list_filtered
    
    return df

In [ ]:
# Function that uses the UniProt API to get the protein Name from the string ID (this is for string db)

def convert_stringID_to_uniprotName(string_id):
  
  uniprot_api_url = "https://rest.uniprot.org/uniprotkb/search?query=gene_exact:"

  query = string_id

  uniprot_request_url = f"{uniprot_api_url}{query}+AND+organism_id:9606"

  uniprot_response = requests.get(uniprot_request_url)
  uniprot_json = uniprot_response.text
  uniprot_dict = json.loads(uniprot_json)

  if "results" in uniprot_dict and len(uniprot_dict["results"]) > 0:
      uniprot_name = uniprot_dict["results"][0]["uniProtkbId"]
  else:
      uniprot_name = None

  return uniprot_name


# example
l = convert_stringID_to_uniprotName("HAVCR1")
print(l)


In [ ]:
# If you have a df with duplicate values for a certain column but not duplicate value for the other columns, this function will make of these rows, one row but retaining the info

def removeDuplicateRow_butRetainInfo (df, rowWithDuplicate, columnRetain1, columnRetain2, columnRetain3, columnRetain4 = None):
    
    def join_columnValues(series):
        return ', '.join(str(value) for value in series)

    columns_to_aggregate = {
        columnRetain1: join_columnValues,
        columnRetain2: join_columnValues,
        columnRetain3: join_columnValues
        }
    
    if columnRetain4 is not None:
        columns_to_aggregate[columnRetain4] = join_columnValues
        
    df_final = df.groupby(rowWithDuplicate).agg(columns_to_aggregate).reset_index()
    
    return df_final

In [ ]:
# make a function for putting the data into the right format for the upsetplot (at the end of the script)

def convert_column_to_list(df, column):
    
    column_list = df[[column]].values.tolist()
    
    def flatten_list(nested_list):
        return list(itertools.chain(*nested_list))

    interactors = flatten_list(column_list)

    return interactors

In [ ]:
# Extracting the data from the BioGRID API

# BioGRID Access Key: caf14dfd9a0b7be447d282c322b8362e

biogrid_api_url = "https://webservice.thebiogrid.org/interactions"

geneList = [convert_uniprotID_uniprotName(query)]

params = {
    "accesskey": "caf14dfd9a0b7be447d282c322b8362e", # need to request
    "additionalIdentifierTypes": "UNIPROT",
    "format": "json",
    "geneList": geneList,
    "taxId": 9606, # human
    "max": 100000
}

response = requests.get(biogrid_api_url, params=params)
biogrid_interactions = response.json()
print(response)
print(biogrid_interactions)

biogrid_data = {}
for interaction_id, interaction in biogrid_interactions.items():
    biogrid_data[interaction_id] = interaction
    biogrid_data[interaction_id]["INTERACTION_ID"] = interaction_id
    
# loading into dataframe

biogrid_df = pd.DataFrame.from_dict(biogrid_data, orient="index")

columns = [
    "INTERACTION_ID",
    "ENTREZ_GENE_A",
    "ENTREZ_GENE_B",
    "OFFICIAL_SYMBOL_A",
    "OFFICIAL_SYMBOL_B",
    "EXPERIMENTAL_SYSTEM",
    "PUBMED_ID",
    "PUBMED_AUTHOR",
    "THROUGHPUT",
    "QUALIFICATIONS"]

biogrid_df = biogrid_df[columns]

biogrid_df.head(5)

In [ ]:
# Cleaning the dataframe
biogrid_df_filter = convert_geneIDs_uniprotIDs(biogrid_df, 'ENTREZ_GENE_A', 'UniprotID_A')
biogrid_df_filter2 = convert_geneIDs_uniprotIDs(biogrid_df, 'ENTREZ_GENE_B', 'UniprotID_B')
biogrid_df_filter2.head(5)

In [ ]:
# Converting the UniprotID to Uniprot Name
biogrid_df_filter2[['UniprotID_A', 'UniprotID_B']] = biogrid_df_filter2[['UniprotID_A', 'UniprotID_B']].map(convert_uniprotID_uniprotName) # takes around 13sec

In [ ]:
biogrid_df_filter2[['UniprotID_A', 'UniprotID_B']] = biogrid_df_filter2[['UniprotID_A', 'UniprotID_B']].map(convert_uniprotID_uniprotName) # takes the same amount of time

In [ ]:
# Converting the UniprotID to Uniprot Name
biogrid_df_filter[['UniprotID_A', 'UniprotID_B']] = biogrid_df_filter[['UniprotID_A', 'UniprotID_B']].map(convert_uniprotID_uniprotName) # takes around 4min

# filter out the columns that are needed
biogrid_df_filter = biogrid_df[['UniprotID_A', 
                                'UniprotID_B', 
                                'EXPERIMENTAL_SYSTEM', 
                                'PUBMED_ID',
                                'PUBMED_AUTHOR']]

# renaming column headers
df_biogrid_final = biogrid_df_filter.rename(columns= {  'UniprotID_A': 'biogrid_interactor_a', 
                                                        'UniprotID_B': 'biogrid_interactor_b',
                                                        'EXPERIMENTAL_SYSTEM': 'biogrid_method',
                                                        'PUBMED_ID': 'biogrid_pubID',
                                                        'PUBMED_AUTHOR': 'biogrid_publication'})

# get one column, only the interactor of the query
df_biogrid = get_interactors_for_target(df_biogrid_final, 'biogrid_interactor_a', 'biogrid_interactor_b', query)

# Now we have some duplicate rows based on the 'interactor_of_query' column, so we joining duplicate rows but retaining the information, so converting all the info to one row
df_biogrid = removeDuplicateRow_butRetainInfo(df_biogrid, 'interactor_of_' + query ,'biogrid_publication', 'biogrid_method', 'biogrid_pubID' )

df_biogrid.head(5)

In [ ]:
# Extracting the data from the IntAct API using PSCIQUIC

intact_api_url = "http://www.ebi.ac.uk/Tools/webservices/psicquic/intact/webservices"

version = "current"
method = "interactor"
protein = query
format = "tab25"

intact_request_url = f"{intact_api_url}/{version}/search/{method}/{protein}?format={format}"

intact_raw_data = requests.get(intact_request_url).text

print(intact_raw_data)

In [ ]:
# Getting the data into a pandas df

# Split each row into a list of columns based on PSI-MI TAB 2.5 format
intact_columns = ['Unique identifier for interactor A', 
                    'Unique identifier for interactor B', 
                    'Alternative identifier for interactor A', 
                    'Alternative identifier for interactor B', 
                    'Aliases for A', 
                    'Aliases for B', 
                    'Interaction detection methods', 
                    'First author', 
                    'Identifier of the publication', 
                    'NCBI Taxonomy identifier for interactor A', 
                    'NCBI Taxonomy identifier for interactor B', 
                    'Interaction types', 
                    'Source databases', 
                    'Interaction identifier(s)', 
                    'Confidence score']

intact_rows = [row.split('\t') for row in intact_raw_data.split('\n')]

# Create a pandas DataFrame from the list of rows and columns
intact_df = pd.DataFrame(intact_rows, columns=intact_columns)
intact_df.shape


In [ ]:
# # Cleaning up the dataframe

# filter out the columns that are needed
intact_df_filter = intact_df[['Unique identifier for interactor A', 
                                  'Unique identifier for interactor B', 
                                  'Interaction detection methods', 
                                  'First author', 
                                  'Identifier of the publication', 
                                  'Confidence score']]

# remove the last row (this is a row with no information, an empty row)
intact_df_filter = intact_df_filter[:-1]

# removing the uniprotkbID refix from the name
intact_df_filter[['Unique identifier for interactor A', 'Unique identifier for interactor B']] = intact_df_filter[['Unique identifier for interactor A', 'Unique identifier for interactor B']].map(lambda x: x.removeprefix('uniprotkb:'))

# filtering out rows with IntactID instead of UniprotID
intact_df_filter = intact_df_filter[~intact_df_filter['Unique identifier for interactor B'].str.contains('intact:')]

# apply the convert_protein_ID_name function to the first two rows
intact_df_filter[['Unique identifier for interactor A', 'Unique identifier for interactor B']] = intact_df_filter[['Unique identifier for interactor A', 'Unique identifier for interactor B']].map(convert_uniprotID_uniprotName) # this step takes a while (around 15min)

In [ ]:
# In the interest of time, I splitted the above cell up

# removing exact duplicate rows
df_intact_dupl = intact_df_filter.drop_duplicates()

# rename the column headers, with prefix IntAct
df_intAct_final = df_intact_dupl.rename(columns= {'Unique identifier for interactor A': 'IntAct_interactor_a', 
                                        'Unique identifier for interactor B': 'IntAct_interactor_b',
                                        'Interaction detection methods': 'IntAct_method',
                                        'First author': 'IntAct_publication',
                                        'Identifier of the publication': 'IntAct_pubID',
                                        'Confidence score': 'IntAct_score'})

# get one column, only the interactor of the query
df_IntAct = get_interactors_for_target(df_intAct_final, 'IntAct_interactor_a', 'IntAct_interactor_b', query)

# clean up the IntAct_score column, that it has only the score and the string ('intact-miscore:')
df_IntAct[['IntAct_score']] = df_IntAct[['IntAct_score']].map(lambda x: x.removeprefix('intact-miscore:'))

# Now we have some duplicate rows based on the 'interactor_of_query' column, so we joining duplicate rows but retaining the information, so converting all the info to one row
df_IntAct = removeDuplicateRow_butRetainInfo(df_IntAct, 'interactor_of_' + query, 'IntAct_publication', 'IntAct_method', 'IntAct_pubID', 'IntAct_score')

df_IntAct.head(5)

In [ ]:
# Extracting the data from the STRING-API

string_api_url = "https://string-db.org//api"

output_format = "json"
method = "interaction_partners"

string_request_url = "/".join([string_api_url, output_format, method])

params = {
    "identifiers": query,
    "species": 9606, # human
    "required_score": 400,
    "limit": 1000000 
}

response = requests.post(string_request_url, data = params)

string_raw_data = response.text

print(string_raw_data)

In [ ]:
# Getting the STRING-API data into a pandas df
string_df = pd.read_json(string_raw_data)
string_df.head(5)

In [ ]:
# Cleaning up the string dataframe

string_df_filter_2 = string_df

# Converting the GeneID to UniprotID
string_df_filter_2[['UniprotID_A', 'UniprotID_B']] = string_df_filter_2[['preferredName_A', 'preferredName_B']].map(convert_stringID_to_uniprotName) # takes around 11min


In [ ]:
# filter out the columns that are needed
string_df_filter_3 = string_df_filter_2[[  'UniprotID_A', 
                                        'UniprotID_B', 
                                        'score', 
                                        'escore']]

# renaming column headers
df_string_filter_3 = string_df_filter_3.rename(columns= {'UniprotID_A': 'string_interactor_a', 
                                                    'UniprotID_B': 'string_interactor_b',
                                                    'score': 'string_score',
                                                    'escore': 'string_escore'})

# get one column, only the interactor of the query
df_string_1 = get_interactors_for_target(df_string_filter_3, 'string_interactor_a', 'string_interactor_b', query)

print(type(df_string_1))

# removing duplicates but first remove NaN and sort on score
df_string_1 = df_string_1.dropna(subset = ["interactor_of_" + query])
df_string_1 = df_string_1.sort_values(by='string_escore')

# Drop the duplicate rows based on the 'id' and 'age' columns.
df_string = df_string_1.drop_duplicates(subset=["interactor_of_" + query])

# filtering the df on the escore
df_string = df_string[df_string['string_escore'] != 0]

df_string.head(5)

In [ ]:
# # GETTING THE DATA FROM the HIPPIE-API

# hippie_api_url = "http://cbdm-01.zdv.uni-mainz.de/~mschaefer/hippie/queryHIPPIE.php"

# protein_to_query = "CADM4"
# layer = 1 #to query protein within input set (0) or against all HIPPIE proteins (1, default)
# threshold = 0 #confidence threshold, default is 0
# format = "conc_file" #this generates a tab seperated text file (other interesting input types: "mitab", "browser")

# hippie_request_url = f"{hippie_api_url}?proteins={protein_to_query}&layers={layer}&conf_thres={threshold}&out_type={format}"

# hippie_response = requests.get(hippie_request_url).text

# print(hippie_response)

# print(hippie_request_url)

# # there seems to be an issue with the PHP request response, probably the server is not correctly configured

In [ ]:
# ## !!!!! HIPPIE website seems to be down at the moment, worked but is not reliable, got 500 errors

# # GETTING THE DATA FROM HIPPIE through WEBSCRAPING
# import requests
# import json
# import re
# from bs4 import BeautifulSoup

# protein = query

# url = "http://cbdm-01.zdv.uni-mainz.de/~mschaefer/hippie/query.php?s="+str(protein)

# payload = {}
# headers = {
# 'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7', 
# 'Accept-Language': 'nl-NL,nl;q=0.9,en-US;q=0.8,en;q=0.7,fr;q=0.6' ,
# 'Connection': 'keep-alive', 
# 'Referer': 'http://cbdm-01.zdv.uni-mainz.de/~mschaefer/hippie/',
# 'Upgrade-Insecure-Requests': '1' ,
# 'User-Agent': 'Mozilla/5.0 (Linux; Android 6.0; Nexus 5 Build/MRA58N) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Mobile Safari/537.36' 
# }

# response = requests.request("GET", url, headers=headers, data=payload)
# soup = BeautifulSoup(response.text, "html.parser")
# table = soup.find('tbody') # already skipped the columns names

# # rows = table.find_all('tr')

# # interactions = []
# # for idx, row in enumerate(rows):
# #     data = row.find_all("td")
# #     interaction = {
# #     "Interactor": data[0].text,
# #     "EntrezGeneID": data[1].text,
# #     "GeneSymbol": data[2].text,
# #     "Score": data[3].text
# #     }
# #     interactions.append(interaction) 
   

# print(response)


In [ ]:
# GETTING THE DATA FROM THE APID db by WEBSCRAPING

import requests
import json
import re
from bs4 import BeautifulSoup


def extract_table(proteinid):
    url = "http://cicblade.dep.usal.es:8080/APID/InteractionsGrid.action?protein1="+str(proteinid)+"&protein2=NA"

    payload = {}
    headers = {
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
    'Accept-Language': 'en-US,en;q=0.9',
    'Connection': 'keep-alive',
    'Cookie': 'JSESSIONID=086030D12C94A8248DA2B5B9A84C16FA; _ga=GA1.2.581619552.1696441740; _gid=GA1.2.818200239.1696441740; _ga_7JSDHY18SK=GS1.2.1696441740.1.1.1696442421.0.0.0',
    'Referer': 'http://cicblade.dep.usal.es:8080/APID/searchProtein.action',
    'Upgrade-Insecure-Requests': '1',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36'
    }

    response = requests.request("GET", url, headers=headers, data=payload)
    soup = BeautifulSoup(response.text, "html.parser")
    table = soup.find('table', id="interactions") #Find the table

    rows = table.find_all("tr") # Find all table rows
    interactions = [] #initialize empty list
    for idx, row in enumerate(rows): #loop over rows, keep index
        if idx == 0: # first row is the header, skip.
            pass
        else:
            try:
                data1 = row.find_all("td") # get column
                interaction = { # build intraction object
                "ProteinA": data1[0].get_text().strip(),
                "ProteinB": data1[1].get_text().strip(),
                "MethodType": data1[2].get_text().strip(),
                "Method": data1[3].get_text().strip(),
                "Publication": re.sub(' +', ' ',data1[4].get_text().strip().replace("\n", "")),
                "Source": data1[5].get_text().strip()
                }
                interactions.append(interaction) #append interaction object to result list
            except:
                pass
    return interactions #return the result list


# To find the UniProtID from the protein name
proteinid = convert_protein_ID_Name(query)
apid_results = extract_table(proteinid)
# print(json.dumps(apid_results, indent=4))
print(apid_results)

In [ ]:
# Getting the data from APID into a df

apid_df = pd.DataFrame(apid_results)

# filter out the columns that are needed
apid_df_filter = apid_df[[  'ProteinA', 
                                'ProteinB', 
                                'Method', 
                                'Publication',
                                'Source']]

# renaming column headers
df_apid_final = apid_df_filter.rename(columns= {'ProteinA': 'apid_interactor_a', 
                                                    'ProteinB': 'apid_interactor_b',
                                                    'Method': 'apid_method',
                                                    'Publication': 'apid_publication',
                                                    'Source': 'apid_source'})

# get one column, only the interactor of the query
df_apid_int = get_interactors_for_target(df_apid_final, 'apid_interactor_a', 'apid_interactor_b', query)


# Now we have some duplicate rows based on the 'interactor_of_query' column, so we joining duplicate rows but retaining the information, so converting all the info to one row
df_apid = removeDuplicateRow_butRetainInfo(df_apid_int, 'interactor_of_' + query, 'apid_method', 'apid_publication', 'apid_source')

df_apid.head(5)

In [ ]:
# Make an intersecions diagram using the pyUpSet

# Put the data into the right format using the custom function
biogrid_interactors = convert_column_to_list(df_biogrid, 'interactor_of_' + query)
IntAct_interactors = convert_column_to_list(df_IntAct, 'interactor_of_' + query)
string_interactors = convert_column_to_list(df_string, 'interactor_of_' + query)
apid_interactors = convert_column_to_list(df_apid, 'interactor_of_' + query)

# Plot the data into an upSetplot
ppis = from_contents({'BioGrid': biogrid_interactors, 'IntAct': IntAct_interactors, 'STRING': string_interactors, 'APID': apid_interactors})
ax_dict = UpSet(ppis, subset_size='count', show_counts=True).plot()

In [ ]:
import collections

list1 = [1, 2, 3, 4, 5, 3, 4]

# Check for duplicates
counter = collections.Counter(string_interactors)
if any(count > 1 for count in counter.values()):
    print("The list contains duplicates.")
else:
    print("The list does not contain duplicates.")

# Extract the duplicate values
duplicate_values = [key for key, count in counter.items() if count > 1]
print(duplicate_values)

print(string_interactors)

# STRING has 2 duplicates

In [ ]:
# merge the separate dataframes together based on one column
dfs = [df_biogrid, df_IntAct, df_string, df_apid]
final_df = reduce(lambda left, right: pd.merge(left,right, on=['interactor_of_' + query], how='outer'), dfs)

In [ ]:
# filter the dataframe based on the subcellular location, using the Uniprot-API

def get_subcellular_location(protein):
    
    uniprot_api_url = "https://rest.uniprot.org/uniprotkb" 
    format = "json"
    uniprot_request_url = f"{uniprot_api_url}/{protein}?format={format}"
    uniprot_response = requests.get(uniprot_request_url)

    uniprot_json = uniprot_response.json()

    subcellular_locations = []
    for comment in uniprot_json.get('comments', []):
        if comment.get('commentType') == 'SUBCELLULAR LOCATION':
            for subcellular_location in comment.get('subcellularLocations', []):
                location_value = subcellular_location.get('location', {}).get('value', '')
                subcellular_locations.append(location_value)
    
    return subcellular_locations

# make a column with the UniprotID from the interactor, this will be used for the subcellular location
# final_df[['UniprotID']] = final_df[['interactor_of_' + query]].map(convert_protein_ID_Name)

# # now go over the dataframe and make a new column with the subcellular location
final_df[['subcellularLocation']] = final_df[['interactor_of_' + query]].map(get_subcellular_location) # takes around 5min


In [ ]:
# filter on the following subcellular locations
subcellular_locations = ["Membrane", "membrane",
                         "Cell junction", "cell junction",
                         "Cell projection", "cell projection",
                         "Cell membrane", "cell membrane",
                         "Plasma membrane", "plasma membrane",
                         "Secreted", "secreted",
                         "Extracellular space", "extracellular space",
                         "Extracellular matrix", "extracellular matrix"
                         "Extracellular exosome", "extracellular exosome",
                         "Cell surface", "cell surface"]

final_df_filtered = final_df[final_df['subcellularLocation'].apply(lambda x: any(location in subcellular_locations for location in x))]

# exporting to a csv
final_df_filtered.to_csv("interactors_of_" + query + ".csv")

In [ ]:
# Add the gene symbols & geneID (now I'm using mouse genes because there are more GO terms for these then human )

# first convert the human uniprotName to human uniprotID, make new column "interactor_of_VCAM1_uniprotID"
final_df_filtered[["interactor_of_" + query + "_uniprotID"]] = final_df_filtered[['interactor_of_' + query]].map(convert_protein_ID_Name)




In [ ]:
# then convert the human uniprotID to human gene symbol
def convert_uniprotID_geneSymbol(UniprotID):
    mygene_api_url = "https://mygene.info/v3/query?q="
    UniprotID = UniprotID
    mygene_request_url =f"{mygene_api_url}{UniprotID}&species=human"
 
    mygene_response = requests.get(mygene_request_url)
    mygene_json = mygene_response.json()   
    
    if "hits" in mygene_json and len(mygene_json["hits"]) > 0 and "symbol" in mygene_json["hits"][0]:
        gene_symbol = mygene_json["hits"][0]["symbol"]
        return gene_symbol
    else:
        return None

# example:
print(convert_uniprotID_geneSymbol("Q13797"))

# now make a new column "interactor_of_VCAM1_HUMAN_geneSymbol"
final_df_filtered[["interactor_of_" + query + "_geneSymbol"]] = final_df_filtered[["interactor_of_" + query + "_uniprotID"]].map(convert_uniprotID_geneSymbol)


In [ ]:
# exporting to a csv
final_df_filtered.to_csv("interactors_of_" + query + ".csv")